# Session 1 · Part 3 — Programming Model: NetQMPI Basics

**14:00 – 14:30**

---

Everything from **notebook 02** carries over. It does not carry over for free: four
properties we took for granted stop being true, and every design decision in
NetQMPI follows from those four breakages.

| MPI assumes | In the quantum case |
|---|---|
| You can copy a value | **No-cloning theorem.** An unknown state cannot be duplicated. `Bcast` is not slow — it is *forbidden*. |
| Reading does not destroy | Measurement collapses the state. There is no "inspect the buffer without consuming it". |
| The channel is reusable | Every transfer **consumes** an entangled pair. Bandwidth is an inventory, not a rate. |
| A late message is still valid | Entanglement **decoheres**. A correct-but-slow protocol can be a wrong protocol. |

One thing that does *not* change: **teleportation needs classical
communication.** Moving a qubit consumes a pre-shared EPR pair, the sender makes
a Bell measurement, and sends **two classical bits**. Without those two bits the
receiver cannot apply the right Pauli correction and holds garbage. A quantum
network is always a double network — a quantum plane distributing entanglement,
and a classical plane carrying measurement outcomes.

## The map from MPI to NetQMPI

Recall the copy / move / combine classification from notebook 02. It predicts
exactly what survives:

| MPI | NetQMPI | Why |
|---|---|---|
| `Send` / `Recv` | `qsend` / `qrecv` | Direct. Teleportation underneath; the original is destroyed, so no-cloning is respected. |
| `Scatter` / `Gather` | `qscatter` / `qgather` | Direct. They **move**, they do not copy. |
| `Bcast` | — | **Impossible.** Its functional role is taken by `expose`, with different semantics: it *spreads* a state, it does not replicate it. |
| `Reduce` | — | Would need a measurement to obtain combinable values, destroying the superposition. |

`qsend(qubit, dest_rank)` hides the entire teleportation protocol: EPR pair
generation, the local Bell measurement, sending two classical bits, and the
Pauli corrections at the receiver. Four steps, one call.

---
## 1 · The programming model

NetQMPI has a **decoupled architecture**: a user-facing SDK that is completely
backend-agnostic, and a Runtime that plugs in a concrete execution engine. The
same unmodified script runs on a quantum-network simulator, on an HPC vQPU
emulator, or on a plain circuit simulator — you change a command-line flag,
nothing else.

| Backend | Flag | What it targets |
|---|---|---|
| NetQASM / SquidASM | `--netqasm` | Low-level quantum-network simulation |
| **CUNQA** | `--cunqa` | **HPC emulation through virtual QPUs — what we use today** |
| Qiskit Aer | `--aer` | Shot-based circuit simulation |
| Qoala | `--qoala` | Quantum-internet node execution environment |

> ⚠ This image ships **only the CUNQA backend** — the NetQASM and Qoala backends were left out deliberately, because they need NetSquid from a credentialed private index.

Three SDK abstractions are all you need:

- **`Environment`** — the local node's context, and a factory for circuits. It
  arrives as the `env` argument of your `main()`.
- **`Circuit`** — a fluent, **index-based** API. You do not hold qubit objects;
  you refer to qubit `0`, qubit `1`, and so on. Operations are *recorded*, not
  executed immediately.
- **`QMPICommunicator`** — `env.comm`. Gives you `rank`, `size`, the neighbour
  helpers, and the communication primitives.

Two structural points that catch people out:

**`with comm:` is where execution happens.** Everything inside the block is
recorded into an operation container; leaving the block is what dispatches it to
the backend. Results are read *after* the block, from `comm.results`.

**Ranks are a ring.** `comm.get_next_rank(rank)` and `comm.get_prev_rank(rank)`
give you your neighbours, wrapping around. That is the idiom the examples use
instead of hard-coding `0` and `1`.

Run this first and read the output carefully. It tells you the exact flags this
build expects.

In [1]:
# EXAMPLE — what does the installed netqmpi understand?
!netqmpi --help

usage: netqmpi [-h] -n NUM_PROCS [--netqasm | --cunqa | --aer | --qoala]
               [--shots SHOTS] [--config CONFIG]
               script

Run a NetQMPI Python code.

positional arguments:
  script                Path to the NetQMPI Python script to be executed

options:
  -h, --help            show this help message and exit
  -n NUM_PROCS, --num-procs NUM_PROCS
                        Number of parallel processes
  --netqasm             Use NetQASM backend
  --cunqa               Use CUNQA backend
  --aer                 Use Qiskit AerSimulator backend
  --qoala               Use Qoala backend (simulation only)
  --shots SHOTS         Number of shots (overrides the value in --config, if
                        any)
  --config CONFIG       Path to a YAML config file with generic settings and
                        an optional per-backend block (e.g. a 'qoala:' block
                        with link_fidelity and a 'hardware' qdevice section).
                        Replaces pe

---
## 2 · Raising virtual QPUs

This is where the Slurm installation from notebook 01 stops being scaffolding.

CUNQA emulates a **virtual QPU** as a bundle of classical resources plus a
simulation backend. Because a vQPU *is* a classical resource in an HPC
environment, its lifecycle commands are implemented as Slurm wrappers: you must
reserve the resources before your program runs. A run needs **one vQPU
per rank**, and by default expects them to be **already raised** — so one
allocation serves many runs. Each `qraise` deploys a group of
vQPUs sharing one configuration, called a **family**.

The consequence for us is a three-step workflow:

1. **`qraise`** — deploy N vQPUs (a Slurm job is submitted)
2. **run** your NetQMPI program against them
3. **`qdrop`** — release them

In [5]:
!qraise --help

Welcome to qraise command, a command responsible for turning on the required QPUs.

Usage: qraise  [options...]

Options:
    -n,--num-qpus : Number of QPUs to be raised. [default: 0]
        -t,--time : Time for the QPUs to be raised. [default: ]
-c,--cores-per-qpu : Number of cores per QPU. [default: 2]
   -p,--partition : Partition requested for the QPUs. [default: none]
--mem,--mem-per-qpu : Memory given to each QPU in GB. [default: 15]
     -N,--n-nodes : Number of nodes. [default: 1]
       --nodelist : List of nodes where the QPUs will be deployed. [default: none]
--qpuN,--qpus-per-node : Number of qpus in each node. [default: none]
     -b,--backend : Path to the backend config file. [default: none]
--sim,--simulator : Simulator reponsible of running the simulations. [default: Aer]
         --family : Name that identifies which QPUs were raised together. [default: default]
     --co-located : co-located mode. The user can connect with any deployed QPU. [implicit: "true", defaul

In [38]:
# Free resources
!qdrop --all

# Deploy two virtual QPUs, co-located on this single node.
from cunqa.qpu import qraise, get_QPUs

family = qraise(2, "00:10:00", simulator="Aer", co_located=True, quantum_comm=True)
print("family:", family)

Requested QPUs with command:
	qraise -n 2 -t 00:10:00 --quantum_comm --simulator=Aer --co-located
QPUs ready to work ✅
family: 8


In [12]:
# They are scheduler jobs. Confirm they are running before going further.
!squeue

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
                 6     debug   qraise     root  R       0:15      1 e6d6f0516bc7


> ### ⚠ The one setting that will ruin your afternoon
>
> The executor simulates the **whole family in one register**, spanning every
> qubit each vQPU declares, whether the circuits use it or not. So the cost of a
> run is set by `num_qubits` × number of ranks — *not* by your circuit.
>
> With a **statevector** simulator that register is 2^N amplitudes:
>
> | vQPU definition | `-n 2` | `-n 3` | `-n 4` |
> |---|---|---|---|
> | 8 qubits each | 1 MiB | 256 MiB | **64 GiB** |
> | 5 qubits each | 16 KiB | 512 KiB | 16 MiB |
>
> If a cell below hangs, this is the first thing to check.

---
## 3 · Point-to-point: `qsend` and `qrecv`

Compare this with `p2p_demo.py` from notebook 02. Same rank-based branching,
same send/receive pair. That symmetry is deliberate: **if you can read MPI, you
can read this.**

In [16]:
%%writefile qp2p_demo.py
from netqmpi.sdk.environment import Environment


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    next_rank = comm.get_next_rank(rank)
    previous_rank = comm.get_prev_rank(rank)

    with comm:                    # leaving this block runs it on the backend
        circuit = env.create_circuit(num_qubits=1, num_clbits=1)
        if rank == 0:
            circuit.h(0)                          # prepare |+>
            comm.qsend(circuit, [0], next_rank)   # teleport qubit 0
        else:
            comm.qrecv(circuit, [0], previous_rank)
            circuit.measure(0, 0)

    results = comm.results

    if rank != 0:
        print(f"measure: {results}", flush=True)
    else:
        print("teleportation complete", flush=True)

Overwriting qp2p_demo.py


In [19]:
!netqmpi -n 2 qp2p_demo.py --cunqa --shots 1024

teleportation complete
measure: {0: {}, 1: {'0 ': 529, '1 ': 495}}


**What those two lines hid.** `qsend` performed: EPR pair generation with the
neighbour, a local Bell measurement, transmission of two classical bits, and on
the receiving side the conditional `X` and `Z` corrections. The original qubit at
rank 0 no longer exists — the measurement destroyed it. The state *moved*.

We sent |+⟩, so rank 1's counts should be roughly half `0` and half `1` over
1024 shots.

**Try it with a known state instead.** Change `circuit.h(0)` to `circuit.x(0)`
and re-run: rank 1 should now measure `1` on essentially every shot. That is a
much better test of whether teleportation actually worked — a 50/50 result is
also what you would get from a broken channel returning noise.

### Exercise 4 — A distributed Bell state between two ranks

Build $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ with **one
qubit on each rank**.

The recipe is the familiar one — Hadamard, then CNOT — with a twist: the two
halves must end up on different ranks. Build the pair *locally* first, then
teleport one half across.

In [ ]:
%%writefile ex4_bell.py
from netqmpi.sdk.environment import Environment


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    next_rank = comm.get_next_rank(rank)
    previous_rank = comm.get_prev_rank(rank)

    with comm:
        if rank == 0:
            # TODO 1: this rank needs TWO qubits and two classical bits.
            circuit = None   # TODO

            # TODO 2: build the Bell pair locally.
            #         Hadamard on qubit 0, then CNOT control 0, target 1.

            # TODO 3: teleport the SECOND half to the next rank, keeping the first.
            #         After this line the entanglement spans two ranks.

            circuit.measure(0, 0)
        else:
            circuit = env.create_circuit(num_qubits=1, num_clbits=1)

            # TODO 4: receive the half that rank 0 sent, onto local qubit 0.

            circuit.measure(0, 0)

    print(f"rank {rank}: {comm.results}", flush=True)

In [28]:
%%writefile ex4_bell.py
from netqmpi.sdk.environment import Environment


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    next_rank = comm.get_next_rank(rank)
    previous_rank = comm.get_prev_rank(rank)

    with comm:
        if rank == 0:
            # TODO 1: this rank needs TWO qubits and two classical bits.
            circuit = env.create_circuit(num_qubits=2, num_clbits=2)   # TODO

            # TODO 2: build the Bell pair locally.
            #         Hadamard on qubit 0, then CNOT control 0, target 1.
            circuit.h(0).cx(0,1)
            
            # TODO 3: teleport the SECOND half to the next rank, keeping the first.
            #         After this line the entanglement spans two ranks.
            comm.qsend(circuit, [1], next_rank)
            
            circuit.measure(0, 0)
        else:
            circuit = env.create_circuit(num_qubits=1, num_clbits=1)

            # TODO 4: receive the half that rank 0 sent, onto local qubit 0.
            comm.qrecv(circuit, [0], previous_rank)
            
            circuit.measure(0, 0)

    print(f"rank {rank}: {comm.results}", flush=True)

Overwriting ex4_bell.py


In [29]:
!netqmpi -n 2 ex4_bell.py --cunqa --shots 1024

rank 0: {}
rank 1: {0: {'0 ': 525, '1 ': 499}, 1: {'0 ': 525, '1 ': 499}}


**Read the output carefully, because it is a trap.**

Each rank reports roughly 50% `0` and 50% `1`. That is *exactly* what you would
see if the two qubits were independent coin flips. A marginal distribution
cannot tell a Bell state from two unrelated random bits.

This is not a limitation of the exercise — it is a real and important property of
distributed entanglement: **the correlation lives in the joint distribution, and
neither node holds it.** To observe it you must either bring the two halves back
together, or do classical post-processing on shot-matched outcomes from both
ranks. The solutions notebook has a variant that teleports *both* halves to rank
1, whose counts then show only `00` and `11` and never `01` or `10`.

In [30]:
%%writefile sol_ex4_bell_joint.py
from netqmpi.sdk.environment import Environment


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    next_rank = comm.get_next_rank(rank)
    previous_rank = comm.get_prev_rank(rank)

    with comm:
        if rank == 0:
            circuit = env.create_circuit(num_qubits=2, num_clbits=2)
            circuit.h(0)
            circuit.cx(0, 1)
            comm.qsend(circuit, [1], next_rank)    # first half
            comm.qsend(circuit, [0], next_rank)    # and the second one too
        else:
            circuit = env.create_circuit(num_qubits=2, num_clbits=2)
            comm.qrecv(circuit, [0], previous_rank)
            comm.qrecv(circuit, [1], previous_rank)
            circuit.measure(0, 0)
            circuit.measure(1, 1)

    if rank != 0:
        print(f"joint counts: {comm.results}", flush=True)

Overwriting sol_ex4_bell_joint.py


In [31]:
!netqmpi -n 2 sol_ex4_bell_joint.py --cunqa --shots 1024

joint counts: {0: {}, 1: {'00 ': 508, '11 ': 516}}


---
## 4 · Non-local operations and distributed gates

Moving a qubit is not always what a distributed algorithm needs. Often several
ranks only want to apply gates **controlled by a remote qubit** — the crossing
rotations of a QFT, for instance. Teleporting the control over and back costs two
transfers and drags any entanglement it already has with it.

The alternative is **telegate**: the qubit stays where it is and is *lent* to the
others through a shared GHZ state.

`expose` and `unexpose` are **collective**, like `MPI_Bcast`: every rank of the
window must call them, and `root` names the rank lending the qubit. The call
returns **the index each rank must use as the control** — its own data qubit on
the root, a freshly reserved communication qubit on every receiver — so the gate
itself is written exactly like a local one.

> **`unexpose` is mandatory, not polite.** Communication qubits and the classical
> bits carrying the protocol corrections are reserved when a window opens and
> released when it closes. Think of the pair as acquire/release on a lock, not as
> a broadcast.

In [43]:
%%writefile expose_demo.py
from netqmpi.sdk.environment import Environment

def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    with comm:
        circuit = env.create_circuit(num_qubits=1, num_clbits=1)

        # Rank 0 expose its qubit 0 and rank 1 receives the index to use.
        control = comm.expose(circuit, 0, [0,1], root=0)

        if rank == 0:
            circuit.h(0)
        elif rank == 1:
            circuit.cx(control, 0)     # controlled-X driven by a REMOTE qubit

        comm.unexpose(circuit, [0,1], root=0)   # the control goes back untouched

        circuit.measure(0, 0)

    print(f"rank {rank}: {comm.results}", flush=True)

Overwriting expose_demo.py


In [44]:
!netqmpi -n 2 expose_demo.py --cunqa --shots 1024

rank 0: {}
rank 1: {0: {'0 ': 511, '1 ': 513}, 1: {'0 ': 1024}}


### Exercise 5 — A non-local CNOT across two QPUs

Apply `CNOT(control = a qubit on rank 0, target = a qubit on rank 1)` **without
moving either data qubit**.

Unlike exercise 4, this one is *directly verifiable*. Put the control in a
definite state: if rank 0's control is |1⟩, rank 1's target must flip to |1⟩ on
every shot. If the control is |0⟩, the target stays at `0`. No 50/50 ambiguity —
the answer is either right or wrong.

In [54]:
%%writefile sol_ex5_nonlocal_cnot.py
from netqmpi.sdk.environment import Environment

ROOT = 0


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    with comm:
        circuit = env.create_circuit(num_qubits=1, num_clbits=1)

        if rank == ROOT:
            circuit.x(0)                              # control = |1>

        # ROOT lends qubit 0; every rank gets the index to use as control.
        control = comm.expose(circuit, 0, list(range(comm.size)), root=ROOT)

        if rank != ROOT:
            circuit.cx(control, 0)                    # target = local qubit 0

        comm.unexpose(circuit, list(range(comm.size)), root=ROOT)        # mandatory release

        circuit.measure(0, 0)

    print(f"rank {rank}: {comm.results}", flush=True)

Overwriting sol_ex5_nonlocal_cnot.py


In [58]:
!netqmpi -n 2 sol_ex5_nonlocal_cnot.py --cunqa --shots 1024

rank 0: {}
rank 1: {0: {'1 ': 1024}, 1: {'1 ': 1024}}


**How to know it worked.** Rank 1 must report `1` on essentially every shot.
Then delete the `circuit.x(0)` line and re-run: rank 1 must report `0` on
essentially every shot. The target follows a control that is on another machine
and never moved.

**Why bother, if exercise 4 reached a similar state?** Two reasons, and both cut
the same way:

- Exercise 4 **moved** a qubit. If that qubit had been entangled with a local
  computation, moving it would have dragged all of that along.
- Exercise 5 **generalises**. `expose` lends to the whole communicator, not to
  one partner. The same source with `-n 3` gives you a three-party controlled
  operation — change one number on the command line, touch nothing in the file.

That is the SPMD promise from notebook 02, holding all the way up through the
quantum layer.

---
## 5 · Release the resources

vQPUs are scheduler allocations. Leaving them up holds the reservation.

In [59]:
!qdrop --all
!squeue

Removed job(s) with ID(s): 9 
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
                 9     debug   qraise     root CG       0:38      1 e6d6f0516bc7


---
## If something failed

NetQMPI ships a directory of programs that are **meant** to fail, one failure
mode each, with a runner that reports what the library says about every one of
them — and it needs no vQPU at all. It is the fastest way to learn what the error
messages mean:

```bash
python examples/frequent_errors/run_all.py
```

Quick triage for today:

| Symptom | Likely cause |
|---|---|
| "no vQPUs up", run stops before building | `qraise` was never run, or the family was dropped |
| Run never finishes | Statevector simulator on oversized vQPUs — see the warning in section 2 |
| Hangs with some ranks idle | Fewer vQPUs raised than ranks requested |
| `co_located` mismatch | The flag must match how the vQPUs were raised |